# Step 5 (fixed) — OCR each crop individually to guarantee box-text alignment

## What was wrong before

The pipeline OCR'd the entire column as one block of text, then paired the resulting lines to OpenCV-detected boxes by position. Whenever the number of OCR'd lines didn't exactly match the number of detected boxes (which happens constantly — some entries span 1 line, some 2, some 3), the pairing drifted and crops no longer matched their text.

## The fix — researched from real historical-document OCR pipelines

The standard approach (used by OCR-D, National Library of Luxembourg's newspaper pipeline, and others) is to never detect boxes and text as two separate processes. Instead: **OCR each box's own cropped image individually.** The text that comes back is guaranteed to belong to that exact crop because it was read from that exact image region — there is no pairing step left to go wrong.

This costs more Gemini calls (one per entry instead of one per column) but eliminates the alignment problem completely.

## Also fixed

- Blank fields no longer render as "nan" in the review HTML
- Entry numbering (Entry N of Total) in the review display
- Overlap-merge pass on detected boxes, so large/bold text (e.g. business ads like Achenbach & Schulte) no longer fires two overlapping boxes over the same block of text, which previously produced duplicate, garbled entries
- Takes a multi-page PDF directly as input (renders pages to images itself) instead of requiring pre-exported PNGs per page
- Produces ONE combined review HTML for the whole run, with Prev/Next page navigation built in, instead of a separate file per page
- Retries transient Gemini server errors automatically, and a crash partway through an 80-page run no longer loses the pages already done -- rerunning the loop picks up where it left off instead of starting over

## 0. Setup

In [1]:
import os
import re
import json
import time
import difflib
from pathlib import Path
from typing import Optional, Literal, List, Dict

import cv2
import fitz  # PyMuPDF -- pip install pymupdf
from tenacity import retry, stop_after_attempt, wait_exponential
import numpy as np
import pandas as pd
from PIL import Image
from pydantic import BaseModel
from google import genai
from google.genai import types

KEY_FILE = Path("api_key.txt")
API_KEY = (
    KEY_FILE.read_text().strip()
    if KEY_FILE.exists()
    else os.getenv("GEMINI_API_KEY", "")
)
assert API_KEY, "API key required"

MODEL_NAME = "gemini-2.5-flash"
client = genai.Client(api_key=API_KEY)

GT_PATH = Path("output_ground_truth") / "ground_truth.json"
assert GT_PATH.exists(), f"Ground truth not found: {GT_PATH}"

OUTPUT_DIR = Path("step5_new_pages_output")
OUTPUT_DIR.mkdir(exist_ok=True)
(OUTPUT_DIR / "csv").mkdir(exist_ok=True)
(OUTPUT_DIR / "excel").mkdir(exist_ok=True)
(OUTPUT_DIR / "overlays").mkdir(exist_ok=True)
(OUTPUT_DIR / "entry_crops").mkdir(exist_ok=True)
(OUTPUT_DIR / "review").mkdir(exist_ok=True)

print("Setup OK")
print(f"Output folder: {OUTPUT_DIR}")

Setup OK
Output folder: step5_new_pages_output


---
## 1. Load page 200 ground truth as few-shot examples

In [2]:
gt_data = json.loads(GT_PATH.read_text(encoding="utf-8"))
gt_boxes = gt_data["boxes"]

all_gt_entries = []
for gb in gt_boxes:
    if gb["skipped"]:
        continue
    for entry in gb["entries"]:
        all_gt_entries.append(entry)

def has_field(e, f):
    return bool(e.get(f, ""))

categories = {
    "rooms": [e for e in all_gt_entries if has_field(e, "rooms")],
    "boarding": [e for e in all_gt_entries if has_field(e, "boarding")],
    "race": [e for e in all_gt_entries if has_field(e, "race")],
    "workplace": [e for e in all_gt_entries if has_field(e, "workplace_address")],
    "ownership": [e for e in all_gt_entries if has_field(e, "ownership")],
    "employer": [e for e in all_gt_entries if has_field(e, "employer")],
    "business": [e for e in all_gt_entries if e.get("entry_type") == "business"],
    "institution": [e for e in all_gt_entries if e.get("entry_type") == "institution"],
    "crossref": [e for e in all_gt_entries if e.get("entry_type") == "cross_reference"],
    "notes": [e for e in all_gt_entries if has_field(e, "notes")],
}

selected_examples = []
seen = set()
for cat_name in categories:
    for entry in categories[cat_name]:
        key = json.dumps(entry, sort_keys=True)
        if key not in seen:
            selected_examples.append(entry)
            seen.add(key)
            break

print(f"Selected {len(selected_examples)} few-shot examples from page 200 ground truth")

Selected 9 few-shot examples from page 200 ground truth


---
## 2. Column and entry-box detection (3rd revision -- small dedup fix)

Same per-row hanging-indent approach as the last revision (that part is
sound). Added one more fix visible in the last review HTML: the Achenbach
entry was split into three fragments -- "...r." / "...r. 1018 LaBranch...
Phones" / "703-5 rings. ACHENBACH..." -- because the smoothed indent signal
can still flicker for a row or two right at a line transition, firing two
"start" detections a few pixels apart for what is really one line break.
Starts within an implausibly small gap of each other are now merged.

**This does not fully solve entry boundaries on its own** -- see section 3
below for the real fix for missing/merged entries.

In [3]:
def preprocess(path):
    img = cv2.imread(str(path))
    if img is None:
        raise FileNotFoundError(path)
    return cv2.fastNlMeansDenoisingColored(img, None, 7, 7, 7, 21)


def detect_column_edges(img):
    h, w = img.shape[:2]
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    _, binary = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    side_margin = int(w * 0.08)
    top_margin = int(h * 0.08)
    bottom_margin = int(h * 0.92)
    core = binary[top_margin:bottom_margin, side_margin:w - side_margin]
    vproj = core.sum(axis=0)
    win = max(5, core.shape[1] // 60)
    smoothed = np.convolve(vproj, np.ones(win) / win, mode="same")
    mid_start, mid_end = len(smoothed) // 3, 2 * len(smoothed) // 3
    gap_idx = mid_start + int(np.argmin(smoothed[mid_start:mid_end]))
    gap_threshold = smoothed.max() * 0.05
    gap_left = gap_idx
    while gap_left > 0 and smoothed[gap_left] < gap_threshold:
        gap_left -= 1
    gap_right = gap_idx
    while gap_right < len(smoothed) - 1 and smoothed[gap_right] < gap_threshold:
        gap_right += 1
    return {
        "left_col":  (side_margin, side_margin + gap_left),
        "right_col": (side_margin + gap_right, w - side_margin),
        "top": top_margin, "bottom": bottom_margin,
    }


def detect_entries_in_column(col_img, smooth_window=7, min_ink_px=2):
    """Per-row hanging-indent detection, plus a fix for the near-duplicate
    starts seen at the transition into a new line (e.g. the Achenbach entry
    getting split into "...r." / "...r. 1018 LaBranch...Phones" / "703-5
    rings. ACHENBACH..." -- three fragments of one entry). The rolling
    median smooths jitter but can still flicker for a row or two exactly at
    a transition, firing two starts a few pixels apart for what is really
    one line break. `min_gap` merges any start that lands implausibly close
    to the previous one.
    """
    gray = cv2.cvtColor(col_img, cv2.COLOR_BGR2GRAY)
    _, binary = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    h, w = binary.shape

    row_ink = (binary > 0).sum(axis=1)
    leftmost_raw = np.full(h, w, dtype=np.int32)
    for y in range(h):
        if row_ink[y] >= min_ink_px:
            nz = np.where(binary[y, :] > 0)[0]
            leftmost_raw[y] = nz[0]

    leftmost = leftmost_raw.copy()
    half = smooth_window // 2
    for y in range(h):
        lo, hi = max(0, y - half), min(h, y + half + 1)
        window = leftmost_raw[lo:hi]
        finite = window[window < w]
        leftmost[y] = int(np.median(finite)) if len(finite) else w

    text_rows = leftmost[leftmost < w]
    if len(text_rows) < 10:
        return []

    left_margin = int(np.median(text_rows))
    tol = max(6, int(w * 0.02))
    at_margin = (leftmost <= left_margin + tol) & (leftmost < w)
    is_indented = (leftmost > left_margin + tol) & (leftmost < w)
    is_empty = (leftmost == w)

    raw_starts = [y for y in range(h)
                  if at_margin[y] and (y == 0 or is_empty[y - 1] or is_indented[y - 1])]
    if not raw_starts:
        return []

    # Merge starts that land right on top of each other (transition jitter,
    # not a real second entry). A real new line is at least a good fraction
    # of a text row's height away from the previous one.
    min_gap = max(10, int(h * 0.006))
    starts = [raw_starts[0]]
    for s in raw_starts[1:]:
        if s - starts[-1] >= min_gap:
            starts.append(s)

    entries = []
    for i, s in enumerate(starts):
        e = starts[i + 1] if i + 1 < len(starts) else h
        while e > s and is_empty[e - 1]:
            e -= 1
        if e - s >= max(10, int(h * 0.004)):
            entries.append((s, e))
    return entries


def merge_overlapping_boxes(boxes, overlap_thresh=0.3):
    """Merge boxes whose vertical spans overlap by more than `overlap_thresh`
    of the shorter box's height, within one column.

    This targets the failure Sean and Norie both flagged from the review
    pages: large/bold ad text (e.g. the Achenbach & Schulte listing) throws
    off the per-row indentation read in detect_entries_in_column, which can
    fire two or more overlapping "starts" over what is really one physical
    block of text. Each overlapping box then gets OCR'd separately, which
    is why the same entry showed up twice with slightly different (garbled)
    readings. Collapsing overlapping boxes into one before OCR runs means
    that block gets read once, as one crop, instead of several times.

    overlap_thresh=0.3: if two boxes share more than 30% of the shorter
    box's height, they're treated as duplicate detections of the same text
    and unioned into one box. Raise toward 0.5 if real adjacent entries
    start getting wrongly merged; lower toward 0.15-0.2 if a known
    duplicate case (like Achenbach & Schulte) still comes through as two
    boxes.
    """
    if not boxes:
        return boxes
    boxes_sorted = sorted(boxes, key=lambda b: b["y1"])
    merged = [dict(boxes_sorted[0])]
    for b in boxes_sorted[1:]:
        last = merged[-1]
        overlap = min(last["y2"], b["y2"]) - max(last["y1"], b["y1"])
        shorter_height = min(last["y2"] - last["y1"], b["y2"] - b["y1"])
        if shorter_height > 0 and overlap / shorter_height > overlap_thresh:
            last["x1"] = min(last["x1"], b["x1"])
            last["y1"] = min(last["y1"], b["y1"])
            last["x2"] = max(last["x2"], b["x2"])
            last["y2"] = max(last["y2"], b["y2"])
        else:
            merged.append(dict(b))
    return merged


def detect_all_entries_fullwidth(image_path):
    img = preprocess(image_path)
    col_edges = detect_column_edges(img)
    pad_y = 2
    boxes = []
    for col_label, (cx_start, cx_end) in [("L", col_edges["left_col"]), ("R", col_edges["right_col"])]:
        top, bottom = col_edges["top"], col_edges["bottom"]
        col_img = img[top:bottom, cx_start:cx_end]
        col_boxes = []
        for (local_y1, local_y2) in detect_entries_in_column(col_img):
            col_boxes.append({
                "col": col_label,
                "x1": cx_start, "y1": max(0, top + local_y1 - pad_y),
                "x2": cx_end, "y2": min(img.shape[0], top + local_y2 + pad_y),
            })
        # Overlap-merge pass -- collapses boxes that got double-detected
        # over the same large/bold text block before ids are assigned.
        # Must run per column so left/right column boxes never merge
        # with each other.
        col_boxes = merge_overlapping_boxes(col_boxes, overlap_thresh=0.3)
        boxes.extend(col_boxes)
    for i, b in enumerate(boxes):
        b["id"] = i
    return img, boxes, col_edges


---
## 3. OCR each crop individually -- alignment fix + entry-count safety net

Per-crop OCR still guarantees the text belongs to that exact crop (no
line-to-box pairing step to drift). On top of that, `ocr_crop` now returns a
**list** of entries instead of one string.

Why: the last run had two crops that swallowed ~15 entries each (box
detection failed across large stretches of the right column). Gemini's
transcription of those crops was actually fine -- it correctly read every
name in the blob -- but the pipeline only parsed and kept the *first*
entry's fields, and the other ~14 people on that crop never made it into the
data at all. That is a real, silent loss, and it's the priority to close
before worrying about box boundaries being pixel-perfect.

Now, if a crop contains multiple stacked entries, Gemini splits them and
every one gets parsed into its own row. A crop with one entry (the normal
case) just returns a one-item list, so nothing changes for correctly
detected boxes.

In [4]:
CROP_OCR_PROMPT = """This image shows one or more entries from a 1900-1901 Houston city directory column.

Most crops contain exactly ONE entry. Sometimes box detection merges more than
one entry into a single crop -- if that happened here, you will see more than
one complete, separate entry stacked in this image (each one starts fresh with
a new person's or business's name back at the left margin, not indented).

Transcribe the complete text. If there is more than one entry visible, split
them and return one string per entry -- do not merge separate people/businesses
into a single string, and do not split a single entry's wrapped lines apart.

Rules:
- Preserve abbreviations exactly: r., h., bds, rms, hhldr, (c), (col), wid, propr, clk, lab, engr, wks
- Preserve commas and periods
- Do NOT expand abbreviations or correct spelling
- If one entry wraps across multiple lines, join those lines into ONE string
- Write [?] for unclear characters
- If the image is blank, an advertisement, or not a real directory entry, return an empty list

Return ONLY a JSON array of strings, nothing else.
Example with one entry: ["Smith John, lab, r 100 Main."]
Example with a merged crop containing three entries: ["Smith John, lab, r 100 Main.", "Smith Robert (c), h. 200 Elm.", "Smith William, clk Foster Co., bds 300 Oak."]"""


@retry(stop=stop_after_attempt(5), wait=wait_exponential(multiplier=2, min=5, max=60))
def ocr_crop(crop_img) -> List[str]:
    """OCR a single crop. Normally returns a list with ONE string.

    This is the safety net for imperfect box detection: if a box ends up
    containing multiple merged entries (as happened for two large stretches
    of column on the last run), the old single-string OCR would transcribe
    the whole blob correctly but the pipeline only kept ONE parsed entry
    from it -- every other real person on that crop was silently dropped
    from the data. Asking Gemini to return a JSON list of entries means a
    bad box degrades to "extra rows to review" instead of "lost data",
    which matters more than getting every box boundary pixel-perfect.
    """
    rgb = cv2.cvtColor(crop_img, cv2.COLOR_BGR2RGB)
    pil = Image.fromarray(rgb)
    resp = client.models.generate_content(
        model=MODEL_NAME, contents=[CROP_OCR_PROMPT, pil],
        config=types.GenerateContentConfig(response_mime_type="application/json"),
    )
    try:
        texts = json.loads(resp.text)
        if not isinstance(texts, list):
            texts = [texts]
    except (json.JSONDecodeError, TypeError):
        texts = []
    return [t.strip() for t in texts if isinstance(t, str) and t.strip() and t.strip() != "NOT_AN_ENTRY"]


print("Per-crop OCR function ready (returns a list -- 1 entry normally, more if a crop got merged)")


Per-crop OCR function ready (returns a list -- 1 entry normally, more if a crop got merged)


---
## 4. Parsing (name-order rule + determinism fix)

Two bugs, both visible in the page 2 review: "Addison Wiley" parsed as
last_name="Wiley" (should be "Addison" -- consistent with every other
Addison entry on the same page), and "Ah For (Chinese)" -- the same
literal text, appearing twice because of a split box -- parsed two
different ways in two separate calls.

Cause of the first: the prompt never stated the directory's name-order
convention, so the model fell back on which word "sounds like" a first vs.
last name. Words like Wiley, Tobe, Benjamin, Sheely, Beate read as
plausible first names on their own, so they got flipped even though the
directory's format (surname always first) makes this unambiguous. Added an
explicit rule instead of relying on the model's judgment.

Cause of the second: no temperature was set on the parse call, so identical
input wasn't guaranteed to produce identical output. Set to 0.

In [5]:
class DirectoryEntry(BaseModel):
    is_directory_entry: bool
    not_entry_reason: Optional[str] = None
    entry_type: Literal["person", "business", "institution", "cross_reference", "unclear"] = "person"
    last_name: Optional[str] = None
    first_name: Optional[str] = None
    business_name: Optional[str] = None
    racial_marker_raw: Optional[str] = None
    occupation_raw: Optional[str] = None
    employer: Optional[str] = None
    workplace_address: Optional[str] = None
    residence_raw: Optional[str] = None
    boarding_raw: Optional[str] = None
    rooms_raw: Optional[str] = None
    residence_qualifier: Optional[str] = None
    ownership_type: Optional[Literal["home", "householder"]] = None
    notes: Optional[str] = None


PARSE_PROMPT = """Parse this 1900-1901 Houston directory entry into structured fields.

Entry text: {raw_text}

Rules:
1. NAME ORDER (apply this before anything else, for entry_type="person"):
   this directory always prints SURNAME FIRST, given name(s) second --
   "Addison Wiley" -> last_name="Addison", first_name="Wiley". This holds
   even when the given name happens to look like a surname on its own
   (e.g. "Aflie Tobe" -> last_name="Aflie", NOT last_name="Tobe"; "Albert
   Benjamin" -> last_name="Albert", NOT last_name="Benjamin"). Do not use
   general knowledge about which word is a "more common" first or last
   name to decide -- position in the entry is the only signal: the very
   first name-token is always last_name, the one(s) after it are
   first_name. This rule does not apply to business/institution names or
   ALL-CAPS cross-reference lines.
2. If NOT a real listing, set is_directory_entry=false, leave all fields null.
3. BOARDING vs RESIDENCE vs ROOMS — DIFFERENT fields:
   - "bds 803 Main" -> boarding_raw="bds 803 Main", residence_raw=null
   - "rms over 1219 Hamilton" -> rooms_raw="rms over 1219 Hamilton", residence_raw=null
   - "r. 904 McKee" -> residence_raw="r. 904 McKee"
   - "h. 1109 Fannin" -> residence_raw="h. 1109 Fannin", ownership_type="home"
4. OCCUPATION vs EMPLOYER — split at the boundary:
   - "teamster Hipp & Key" -> occupation_raw="teamster", employer="Hipp & Key"
5. WORKPLACE ADDRESS — separate from residence.
6. RACIAL MARKER — preserve parentheses: "(c)" not "c"
7. OWNERSHIP: "h." -> "home", "hhldr" -> "householder"
8. NOTES — widow status (wid), phone numbers, cross-references, honorifics
9. Copy abbreviations exactly. Use null for missing fields.
"""


def build_prompt(raw_text, examples):
    prompt = PARSE_PROMPT.format(raw_text=raw_text)
    if examples:
        prompt += "\n\nExamples of correctly parsed entries from this directory:\n"
        for ex in examples[:10]:
            fields_str = json.dumps({k: v for k, v in ex.items() if v}, ensure_ascii=False)
            prompt += f"\nCorrect: {fields_str}\n"
    return prompt


@retry(stop=stop_after_attempt(5), wait=wait_exponential(multiplier=2, min=5, max=60))
def parse_entry(raw_text, examples):
    prompt = build_prompt(raw_text, examples)
    resp = client.models.generate_content(
        model=MODEL_NAME, contents=prompt,
        config=types.GenerateContentConfig(
            response_mime_type="application/json",
            response_schema=DirectoryEntry,
            temperature=0,  # same entry text should parse the same way every time --
                            # "Ah For (Chinese)" previously came back as last_name="Ah"
                            # in one call and last_name="For" in another call on the
                            # exact same text
        ),
    )
    return DirectoryEntry.model_validate_json(resp.text)


---
## 5. Full page processor -- crop-by-crop OCR + adjacent-duplicate cleanup

Each box is cropped, OCR'd (returning a list of one-or-more entries if a box
got merged), and every entry is parsed into its own row. Before saving, a
pass drops adjacent rows that are the same person duplicated across a
truncated box and a fuller box next to it (e.g. box detection cutting after
"Adels Abraham G, (H Weingarten & Adels)," and then repeating the same name
in the next box, this time with the full "r. 714 Milam." address). Same last
name + first name, and text overlap on the leading ~40 characters, is
treated as a duplicate; the shorter (less complete) one is dropped.

`merged_crops` and `duplicates_removed` in the stats show how much of this
is happening per page -- both numbers being high on a page is worth
revisiting the box detection for that page specifically, but the data
itself won't be lost or silently merged away either way.

In [6]:
def _normalize_name(s):
    return re.sub(r"[^a-z]", "", str(s or "").lower())


def dedupe_adjacent_entries(rows):
    """Box detection sometimes cuts one wrapped entry into two boxes: a
    truncated box with just the first line, immediately followed by a box
    with the full entry (this happens when a continuation line like
    "r. 714 Milam." isn't indented enough to be recognized as a
    continuation, e.g. the Adels Abraham G. entry showing up twice in a
    row). Since losing an entry is worse than a recoverable duplicate, the
    safety net above intentionally leans toward over-counting -- this pass
    cleans up the specific case where that shows up: the same person twice
    in a row, one entry a near-prefix of the other. Keeps the more complete
    (longer) of the two and drops the truncated one.
    """
    keep = [True] * len(rows)
    for i in range(1, len(rows)):
        if not keep[i - 1]:
            continue
        a, b = rows[i - 1], rows[i]
        last_a, last_b = _normalize_name(a.get("last_name")), _normalize_name(b.get("last_name"))
        first_a, first_b = _normalize_name(a.get("first_name")), _normalize_name(b.get("first_name"))
        if not last_a or last_a != last_b or first_a != first_b:
            continue
        text_a, text_b = str(a.get("raw_entry_text", "")), str(b.get("raw_entry_text", ""))
        overlap = difflib.SequenceMatcher(None, text_a[:40], text_b[:40]).ratio()
        if overlap < 0.6:
            continue
        if len(text_a) <= len(text_b):
            keep[i - 1] = False
        else:
            keep[i] = False
    return [r for r, k in zip(rows, keep) if k]


def process_page(image_path, page_label, gt_examples):
    """Process one page: detect boxes, OCR each crop individually (possibly
    into multiple entries if a crop got merged), parse fields, then drop
    adjacent truncated/full duplicates of the same person."""
    start = time.time()
    print(f"\n== {page_label} ==")

    img, boxes, col_edges = detect_all_entries_fullwidth(image_path)
    H, W = img.shape[:2]
    print(f"  Detected {len(boxes)} entry boxes")

    overlay = img.copy()
    for b in boxes:
        color = (0, 200, 0) if b["id"] % 2 == 0 else (0, 130, 255)
        cv2.rectangle(overlay, (b["x1"], b["y1"]), (b["x2"], b["y2"]), color, 2)
        cv2.putText(overlay, str(b["id"]), (b["x1"] + 4, b["y1"] + 16),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.45, color, 1, cv2.LINE_AA)
    cv2.imwrite(str(OUTPUT_DIR / "overlays" / f"{page_label}_overlay.png"), overlay)

    boxes_sorted = sorted(boxes, key=lambda b: (b["col"], b["y1"]))

    rows = []
    skipped = 0
    merged_crops = 0
    print(f"  OCR-ing {len(boxes_sorted)} crops individually...")

    for i, b in enumerate(boxes_sorted):
        crop = img[b["y1"]:b["y2"], b["x1"]:b["x2"]]
        crop_name = f"{page_label}_{b['col']}_{i:03d}.png"
        crop_path = OUTPUT_DIR / "entry_crops" / crop_name
        cv2.imwrite(str(crop_path), crop)

        try:
            raw_texts = ocr_crop(crop)
        except Exception as e:
            # Retries (see @retry on ocr_crop) already tried 5 times with
            # backoff -- if it still failed, this one crop is lost, but the
            # rest of this page (and every other page) keeps going instead
            # of the whole 80-page run dying on one flaky request.
            print(f"    [{i}] OCR failed after retries, skipping this crop: {e}")
            skipped += 1
            continue
        if not raw_texts:
            skipped += 1
            continue
        if len(raw_texts) > 1:
            merged_crops += 1

        for j, raw_text in enumerate(raw_texts):
            try:
                entry = parse_entry(raw_text, gt_examples)
            except Exception as e:
                print(f"    [{i}.{j}] parse failed: {e}")
                continue

            if not entry.is_directory_entry:
                skipped += 1
                continue

            entry_id = f"{page_label}_{b['col']}_{i:03d}" + (f"_{j}" if len(raw_texts) > 1 else "")
            rows.append({
                "entry_id": entry_id,
                "source_page": page_label,
                "raw_entry_text": raw_text,
                "entry_crop_path": str(crop_path),
                "is_directory_entry": True,
                "entry_type": entry.entry_type,
                "last_name": entry.last_name,
                "first_name": entry.first_name,
                "business_name": entry.business_name,
                "racial_marker_raw": entry.racial_marker_raw,
                "occupation_raw": entry.occupation_raw,
                "employer": entry.employer,
                "workplace_address": entry.workplace_address,
                "residence_raw": entry.residence_raw,
                "boarding_raw": entry.boarding_raw,
                "rooms_raw": entry.rooms_raw,
                "residence_qualifier": entry.residence_qualifier,
                "ownership_type": entry.ownership_type,
                "notes": entry.notes,
            })

        if (i + 1) % 15 == 0:
            print(f"    {i + 1}/{len(boxes_sorted)} crops processed...")

    before_dedupe = len(rows)
    rows = dedupe_adjacent_entries(rows)
    duplicates_removed = before_dedupe - len(rows)

    df = pd.DataFrame(rows)
    csv_path = OUTPUT_DIR / "csv" / f"{page_label}.csv"
    df.to_csv(csv_path, index=False)

    elapsed = round(time.time() - start, 1)
    print(f"  Done: {len(df)} entries, {skipped} skipped, {merged_crops} crops contained >1 entry, "
          f"{duplicates_removed} adjacent duplicates removed, {elapsed}s")
    return df, {"page": page_label, "boxes": len(boxes),
                "entries": len(df), "skipped": skipped,
                "merged_crops": merged_crops,
                "duplicates_removed": duplicates_removed, "time": elapsed}


---
## 6. Run on target pages

In [7]:
# ---- PDF input: render each page of the source PDF to an image, cache it ----
PDF_PATH = Path(r"C:\Users\ABHIRAMI.K\Downloads\1900-1901 - 80 pages.pdf")
PDF_PAGE_LABEL_PREFIX = "1900_p"

# Testing knobs: run a small range first, then widen once you've checked the
# output. 80 pages of Gemini calls is a multi-hour run (4 pages took ~24 min
# total last time, so roughly 6-8 min/page) -- don't jump straight to the
# full 80 without a quick check on 2-3 pages first.
PAGE_START = 1
PAGE_END = 80          # e.g. set to 3 for a quick test run

PDF_PAGES_DIR = Path("pdf_pages_cache")
PDF_PAGES_DIR.mkdir(exist_ok=True)


def render_pdf_pages(pdf_path, out_dir, page_start, page_end, dpi=300):
    """Render pdf_path's pages [page_start, page_end] (1-indexed, inclusive)
    to PNGs in out_dir, skipping any page already rendered on a prior run.
    Returns a list of (image_path, page_label) tuples -- same shape the rest
    of the pipeline already expects from TARGET_PAGES."""
    doc = fitz.open(pdf_path)
    n = doc.page_count
    page_end = min(page_end, n)
    pages = []
    for i in range(page_start, page_end + 1):
        label = f"{PDF_PAGE_LABEL_PREFIX}{i}"
        img_path = out_dir / f"{label}.png"
        if not img_path.exists():
            pix = doc[i - 1].get_pixmap(dpi=dpi)   # fitz pages are 0-indexed
            pix.save(str(img_path))
        pages.append((img_path, label))
    doc.close()
    return pages


assert PDF_PATH.exists(), f"PDF not found: {PDF_PATH}"
TARGET_PAGES = render_pdf_pages(PDF_PATH, PDF_PAGES_DIR, PAGE_START, PAGE_END)

print(f"PDF has {fitz.open(PDF_PATH).page_count} page(s) total; processing "
      f"{len(TARGET_PAGES)} of them ({PAGE_START}-{PAGE_START + len(TARGET_PAGES) - 1})")
print(f"Rendered page images cached in: {PDF_PAGES_DIR.resolve()}")
est_minutes = len(TARGET_PAGES) * 6.5
print(f"Rough time estimate at ~6.5 min/page: {est_minutes:.0f} min "
      f"({est_minutes/60:.1f} hr) for this run -- plan to let it run unattended")


PDF has 80 page(s) total; processing 80 of them (1-80)
Rendered page images cached in: C:\Users\ABHIRAMI.K\Downloads\pdf_pages_cache
Rough time estimate at ~6.5 min/page: 520 min (8.7 hr) for this run -- plan to let it run unattended


---
### 6a. Sanity-check boxes before spending API calls

Box detection is free -- no Gemini calls. Check the overlay against the
source page and confirm each box wraps exactly one entry (no splits, no
merges, no overlap) before running the full per-crop OCR pass below, since
that pass is what costs time and money.

In [8]:
check_path, check_label = TARGET_PAGES[0]
_img, _boxes, _ = detect_all_entries_fullwidth(check_path)
_overlay = _img.copy()
for b in _boxes:
    color = (0, 200, 0) if b["id"] % 2 == 0 else (0, 130, 255)
    cv2.rectangle(_overlay, (b["x1"], b["y1"]), (b["x2"], b["y2"]), color, 2)
    cv2.putText(_overlay, str(b["id"]), (b["x1"] + 4, b["y1"] + 16),
                cv2.FONT_HERSHEY_SIMPLEX, 0.45, color, 1, cv2.LINE_AA)
_preview_path = OUTPUT_DIR / "overlays" / f"{check_label}_boxcheck.png"
cv2.imwrite(str(_preview_path), _overlay)
print(f"Detected {len(_boxes)} boxes on {check_label}")
print(f"Open and inspect before proceeding: {_preview_path.resolve()}")


Detected 45 boxes on 1900_p1
Open and inspect before proceeding: C:\Users\ABHIRAMI.K\Downloads\step5_new_pages_output\overlays\1900_p1_boxcheck.png


In [9]:
all_dfs = []
all_stats = []
stats_path = OUTPUT_DIR / "page_stats.csv"

for page_path, page_label in TARGET_PAGES:
    if not page_path.exists():
        print(f"Skipping missing: {page_path.name}")
        continue

    csv_path = OUTPUT_DIR / "csv" / f"{page_label}.csv"
    if csv_path.exists():
        # Already processed on a prior run of this cell (e.g. before a crash
        # partway through) -- reload instead of re-sending it to Gemini.
        # A page can legitimately finish with 0 real entries (every box
        # rejected as not-a-directory-entry, e.g. an ad or divider page --
        # p10, p11, p24, p64, p65 in the 80-page run), which writes a CSV
        # that exists but has no parseable rows, so this has to tolerate an
        # empty file rather than assume "exists" means "has data".
        try:
            df = pd.read_csv(csv_path)
        except pd.errors.EmptyDataError:
            df = pd.DataFrame()
        print(f"\n== {page_label} == already done, resuming from saved CSV ({len(df)} entries)")
        stats = {"page": page_label, "boxes": None, "entries": len(df),
                  "skipped": None, "merged_crops": None,
                  "duplicates_removed": None, "time": 0, "resumed": True}
    else:
        try:
            df, stats = process_page(page_path, page_label, selected_examples)
        except Exception as e:
            # Last-resort safety net: even if something outside the per-crop
            # try/except goes wrong for this page (bad image, etc.), log it
            # and keep going rather than losing every page after this one.
            print(f"  !! {page_label} failed entirely, skipping this page: {e}")
            continue

    all_dfs.append(df)
    all_stats.append(stats)

    # Save progress after every page, not just at the end -- so a crash on
    # page 45 still leaves a usable stats file for the recovery cell instead
    # of losing the summary for pages 1-44.
    pd.DataFrame(all_stats).to_csv(stats_path, index=False)

if all_dfs:
    combined = pd.concat(all_dfs, ignore_index=True)
    combined.to_csv(OUTPUT_DIR / "csv" / "all_pages_combined.csv", index=False)
    print(f"\nCombined: {len(combined)} entries across {len(all_dfs)} pages")



== 1900_p1 == already done, resuming from saved CSV (86 entries)

== 1900_p2 == already done, resuming from saved CSV (77 entries)

== 1900_p3 == already done, resuming from saved CSV (83 entries)

== 1900_p4 == already done, resuming from saved CSV (83 entries)

== 1900_p5 == already done, resuming from saved CSV (65 entries)

== 1900_p6 == already done, resuming from saved CSV (74 entries)

== 1900_p7 == already done, resuming from saved CSV (67 entries)

== 1900_p8 == already done, resuming from saved CSV (67 entries)

== 1900_p9 == already done, resuming from saved CSV (83 entries)

== 1900_p10 == already done, resuming from saved CSV (0 entries)

== 1900_p11 == already done, resuming from saved CSV (0 entries)

== 1900_p12 == already done, resuming from saved CSV (81 entries)

== 1900_p13 == already done, resuming from saved CSV (91 entries)

== 1900_p14 == already done, resuming from saved CSV (68 entries)

== 1900_p15 == already done, resuming from saved CSV (82 entries)

== 19

---
## 7. Results summary

In [10]:
if all_stats:
    stats_df = pd.DataFrame(all_stats)
    print(stats_df.to_string(index=False))
    print(f"\nTotal entries: {stats_df['entries'].sum()}")
    print(f"Total time:    {stats_df['time'].sum():.0f}s")
    stats_df.to_csv(OUTPUT_DIR / "page_stats.csv", index=False)

    page boxes  entries skipped merged_crops duplicates_removed  time  resumed
 1900_p1  None       86    None         None               None     0     True
 1900_p2  None       77    None         None               None     0     True
 1900_p3  None       83    None         None               None     0     True
 1900_p4  None       83    None         None               None     0     True
 1900_p5  None       65    None         None               None     0     True
 1900_p6  None       74    None         None               None     0     True
 1900_p7  None       67    None         None               None     0     True
 1900_p8  None       67    None         None               None     0     True
 1900_p9  None       83    None         None               None     0     True
1900_p10  None        0    None         None               None     0     True
1900_p11  None        0    None         None               None     0     True
1900_p12  None       81    None         None        

---
## 8. Excel export

---
### Recovery cell -- only needed if you restarted the kernel

The crash below happened *after* all 80 pages had already finished processing and been saved to CSV -- only the Excel-export step failed. If your Jupyter kernel is still the same one from the original run, `all_dfs` / `all_stats` / `TARGET_PAGES` are still in memory and you can **skip this cell** and go straight to the fixed export cell below.

If you restarted the kernel (or are coming back to this later), run cells 1-6 above again first (imports, setup, ground truth, function defs -- all instant, no API calls), then run this cell instead of the main processing loop. It rebuilds everything from what's already saved on disk, so nothing gets re-sent to Gemini and nothing gets re-billed.

In [11]:
stats_df = pd.read_csv(OUTPUT_DIR / "page_stats.csv")
all_stats = stats_df.to_dict("records")

all_dfs = []
for stats in all_stats:
    label = stats["page"]
    csv_path = OUTPUT_DIR / "csv" / f"{label}.csv"
    if csv_path.exists():
        try:
            all_dfs.append(pd.read_csv(csv_path))
        except pd.errors.EmptyDataError:
            pass  # page finished with 0 real entries -- nothing to add

TARGET_PAGES = [
    (PDF_PAGES_DIR / f"{s['page']}.png", s["page"])
    for s in all_stats
]

print(f"Recovered {len(all_stats)} pages, {sum(len(d) for d in all_dfs)} total entries from disk -- no re-processing needed")


Recovered 80 pages, 5621 total entries from disk -- no re-processing needed


In [12]:
try:
    import openpyxl
    has_openpyxl = True
except ImportError:
    has_openpyxl = False
    print("Run: pip install openpyxl")

if has_openpyxl and all_dfs:
    exported = 0
    for stats in all_stats:
        label = stats["page"]
        csv_path = OUTPUT_DIR / "csv" / f"{label}.csv"
        if not csv_path.exists() or csv_path.stat().st_size == 0:
            print(f"  Skipping {label}: no entries on this page (every box was "
                  f"rejected as not a real directory entry) -- nothing to export")
            continue
        try:
            df_page = pd.read_csv(csv_path)
        except pd.errors.EmptyDataError:
            print(f"  Skipping {label}: empty CSV, nothing to export")
            continue
        df_page.to_excel(OUTPUT_DIR / "excel" / f"{label}.xlsx", index=False, sheet_name=label)
        exported += 1

    combined_xlsx = OUTPUT_DIR / "excel" / "all_pages_combined.xlsx"
    with pd.ExcelWriter(combined_xlsx, engine="openpyxl") as writer:
        for stats in all_stats:
            label = stats["page"]
            csv_path = OUTPUT_DIR / "csv" / f"{label}.csv"
            if not csv_path.exists() or csv_path.stat().st_size == 0:
                continue
            try:
                pd.read_csv(csv_path).to_excel(writer, index=False, sheet_name=label[:31])
            except pd.errors.EmptyDataError:
                continue
    print(f"\nExported {exported} per-page Excel files (of {len(all_stats)} pages total)")
    print(f"Combined Excel: {combined_xlsx}")


  Skipping 1900_p10: empty CSV, nothing to export
  Skipping 1900_p11: empty CSV, nothing to export

Exported 78 per-page Excel files (of 80 pages total)
Combined Excel: step5_new_pages_output\excel\all_pages_combined.xlsx


---
## 9. Review HTML — with entry numbers, no NaN, and guaranteed-matching crops

Blank fields are skipped entirely instead of showing "nan". Each entry now shows its position (Entry N of Total).

In [13]:
def clean_val(v):
    """Return empty string for NaN/None, otherwise the string value."""
    if v is None or (isinstance(v, float) and pd.isna(v)):
        return ""
    s = str(v).strip()
    return "" if s.lower() == "nan" else s


def build_review_combined(labeled_dfs, page_image_paths, output_html):
    """One HTML file covering every page processed in this run. A Prev/Next
    + dropdown page selector in the left panel swaps both the page image and
    the entries list on the right via JavaScript -- no separate file per
    page, no reload, and it scales the same way whether the run covered 4
    pages or 80."""
    pages_json_list = []
    for label, df in labeled_dfs:
        img_path = page_image_paths.get(label)
        page_uri = Path(img_path).resolve().as_uri() if img_path and Path(img_path).exists() else ""
        entries = []
        for _, row in df.iterrows():
            crop_path = clean_val(row.get("entry_crop_path", ""))
            crop_uri = Path(crop_path).resolve().as_uri() if crop_path and Path(crop_path).exists() else ""
            entries.append({
                "raw": clean_val(row["raw_entry_text"]), "crop": crop_uri,
                "fields": {
                    "Last name": clean_val(row.get("last_name")),
                    "First name": clean_val(row.get("first_name")),
                    "Business": clean_val(row.get("business_name")),
                    "Race": clean_val(row.get("racial_marker_raw")),
                    "Occupation": clean_val(row.get("occupation_raw")),
                    "Employer": clean_val(row.get("employer")),
                    "Workplace": clean_val(row.get("workplace_address")),
                    "Residence": clean_val(row.get("residence_raw")),
                    "Boarding": clean_val(row.get("boarding_raw")),
                    "Rooms": clean_val(row.get("rooms_raw")),
                    "Ownership": clean_val(row.get("ownership_type")),
                    "Type": clean_val(row.get("entry_type")),
                    "Notes": clean_val(row.get("notes")),
                },
            })
        pages_json_list.append({"label": label, "pageUri": page_uri, "entries": entries})

    pages_json = json.dumps(pages_json_list, ensure_ascii=False)

    html = """<!DOCTYPE html>
<html><head><meta charset='utf-8'><title>Directory Review -- All Pages</title>
<style>
body{font-family:Calibri,sans-serif;margin:0;background:#f4f4f4;font-size:14px}
.layout{display:flex;height:100vh;overflow:hidden}
.left{flex:0 0 42%;background:#eaeaea;padding:10px;display:flex;flex-direction:column}
.controls{background:white;padding:8px;border-radius:6px;margin-bottom:8px;font-size:13px;display:flex;gap:10px;align-items:center;flex-wrap:wrap}
.controls input[type=range]{flex:1;min-width:80px}
.nav{background:white;padding:8px;border-radius:6px;margin-bottom:8px;font-size:13px;display:flex;gap:8px;align-items:center;justify-content:space-between}
.nav button{background:#1D6E5E;color:white;border:none;border-radius:4px;padding:6px 14px;cursor:pointer;font-size:13px}
.nav button:disabled{background:#aaa;cursor:default}
.nav select{flex:1;padding:5px;border-radius:4px;border:1px solid #ccc;font-size:13px}
.viewport{flex:1;background:white;padding:8px;border-radius:6px;overflow:auto}
.page-wrap{position:relative;display:inline-block}
.page-wrap img{display:block;transform-origin:top left}
.right{flex:1;padding:14px 18px;overflow:auto;background:white}
h1{color:#1D6E5E;font-size:18px;margin:0 0 12px 0}
.card{background:#fafafa;border:1px solid #e5e7eb;border-radius:6px;padding:10px 14px;margin-bottom:10px}
.card img{max-width:100%;max-height:80px;border:1px solid #ddd;margin-bottom:6px}
.entry-num{font-size:11px;color:#1D6E5E;font-weight:bold;margin-bottom:4px}
.raw{color:#6B7280;font-style:italic;font-size:12px;margin-bottom:6px;background:#fff;padding:4px 8px;border-left:3px solid #1D6E5E}
.fields{font-size:12px;display:grid;grid-template-columns:100px 1fr;gap:2px 8px}
.fields b{color:#374151}
</style></head><body>
<div class='layout'>
<div class='left'>
  <div class='nav'>
    <button id='prevBtn' onclick='goPage(currentPage-1)'>&larr; Prev</button>
    <select id='pageSelect' onchange='goPage(parseInt(this.value))'></select>
    <button id='nextBtn' onclick='goPage(currentPage+1)'>Next &rarr;</button>
  </div>
  <div class='controls'>Zoom:<input type='range' min='30' max='150' value='55' id='zs'
    oninput='applyZoom(this.value)'><span id='zoomLabel'>55%</span></div>
  <div class='viewport'><div class='page-wrap' id='pw'><img id='pi' src=''></div></div>
</div>
<div class='right'>
<h1 id='pageTitle'></h1>
<div id='cards'></div>
</div></div>
<script>
const PAGES = __PAGES_JSON__;
let currentPage = 0;

function applyZoom(val) {
  const v = val / 100;
  const img = document.getElementById('pi');
  document.getElementById('zoomLabel').textContent = val + '%';
  img.style.transform = 'scale(' + v + ')';
  document.getElementById('pw').style.width = (img.naturalWidth * v) + 'px';
  document.getElementById('pw').style.height = (img.naturalHeight * v) + 'px';
}

function renderCards(entries) {
  const total = entries.length;
  let html = '';
  entries.forEach((e, i) => {
    let fhtml = '';
    for (const [k, v] of Object.entries(e.fields)) {
      if (v) fhtml += '<b>' + k + ':</b><span>' + v + '</span>';
    }
    const cropTag = e.crop ? "<img src='" + e.crop + "'>" : '';
    html += "<div class='card'><div class='entry-num'>Entry " + (i + 1) + " of " + total + "</div>"
          + cropTag + "<div class='raw'>" + e.raw + "</div>"
          + "<div class='fields'>" + fhtml + "</div></div>";
  });
  document.getElementById('cards').innerHTML = html;
}

function goPage(idx) {
  if (idx < 0 || idx >= PAGES.length) return;
  currentPage = idx;
  const p = PAGES[idx];
  document.getElementById('pi').src = p.pageUri;
  document.getElementById('pageTitle').textContent =
    'Page ' + (idx + 1) + ' of ' + PAGES.length + ' -- ' + p.label + ' (' + p.entries.length + ' entries)';
  document.getElementById('pageSelect').value = idx;
  document.getElementById('prevBtn').disabled = (idx === 0);
  document.getElementById('nextBtn').disabled = (idx === PAGES.length - 1);
  renderCards(p.entries);
  applyZoom(document.getElementById('zs').value);
}

const sel = document.getElementById('pageSelect');
PAGES.forEach((p, i) => {
  const opt = document.createElement('option');
  opt.value = i;
  opt.textContent = (i + 1) + '. ' + p.label + ' (' + p.entries.length + ' entries)';
  sel.appendChild(opt);
});

document.addEventListener('keydown', (ev) => {
  if (ev.key === 'ArrowRight') goPage(currentPage + 1);
  if (ev.key === 'ArrowLeft') goPage(currentPage - 1);
});

goPage(0);
</script>
</body></html>"""

    html = html.replace("__PAGES_JSON__", pages_json)
    output_html.write_text(html, encoding="utf-8")
    return output_html


# Build one combined review file covering every page processed in this run
labeled_dfs = []
page_image_map = {}
for page_path, page_label in TARGET_PAGES:
    csv_path = OUTPUT_DIR / "csv" / f"{page_label}.csv"
    if not csv_path.exists():
        continue
    try:
        df = pd.read_csv(csv_path)
    except pd.errors.EmptyDataError:
        continue  # page finished with 0 real entries -- nothing to show for it
    real = df[df["is_directory_entry"] == True]
    labeled_dfs.append((page_label, real))
    page_image_map[page_label] = page_path

combined_html_path = build_review_combined(
    labeled_dfs, page_image_map, OUTPUT_DIR / "review" / "review_all_pages.html"
)
print(f"Combined review file: {combined_html_path.resolve()}")
print(f"Covers {len(labeled_dfs)} pages, {sum(len(df) for _, df in labeled_dfs)} entries total")


Combined review file: C:\Users\ABHIRAMI.K\Downloads\step5_new_pages_output\review\review_all_pages.html
Covers 78 pages, 5621 entries total


---
## 10. What changed and what to check

**The fix:** every crop is now OCR'd individually, so its text is guaranteed to belong to that exact image. There is no longer a "pair boxes to OCR lines by position" step that could drift.

**Trade-off:** more Gemini calls (one per entry instead of one per column), so processing will take noticeably longer per page — expect several minutes per page instead of under a minute. This is the correct trade for guaranteed alignment.

**What to check after running:** open a review HTML — the crop image directly above each entry's raw text should now visually match. If it still doesn't, the box detection itself (not the OCR) is misplacing the crop boundaries, which would need attention in `detect_entries_in_column`.